In [ ]:
import sqlite3
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt 

In [ ]:
df = pd.read_csv("processed_invoices.csv")
print(df.shape)
df.head()

In [ ]:
def create_invoice_risk_label(row):
    
    if abs(row['invoice_dollars'] - row['total_item_dollars']) > 5:
        return 1
        
    if row['avg_receiving_delay'] > 10:
        return 1

    return 0

df["flag_invoice"] = df.apply(create_invoice_risk_label, axis=1)


print(df["flag_invoice"].value_counts())


In [ ]:
df['flag_invoice'].value_counts().plot(kind='bar')

In [ ]:
df.corr()

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(df.iloc[:,1:-1].corr(), annot=True)
plt.show()

In [ ]:
flagged=df[df['flag_invoice']==1]
normal=df[df['flag_invoice']==0]

In [ ]:
significant_features = []
non_significant_features = []
results = []

In [ ]:
metrics=['invoice_quantity','invoice_dollars','Freight','days_po_to_invoice','days_to_pay','total_brands','total_item_quantity','total_item_dollars','avg_receiving_delay']

In [ ]:
from scipy.stats import ttest_ind
for metric in metrics:
    flagged_mean=flagged[metric].mean()
    normal_mean=normal[metric].mean()
    t_start,p_value=ttest_ind(
        flagged[metric].dropna(),
        normal[metric].dropna(),
        equal_var=False
    )
    if p_value <0.05:
        significant_features.append(metric)
        results.append({
            "metric":metric,
            "flagged_mean":flagged_mean.round(2),
            "normal_mean":normal_mean.round(2),
            "p_value":p_value.round(3)
        })
    else:
        non_significant_features.append(metric)

In [ ]:
non_significant_features

In [ ]:
significant_features

In [ ]:
results

In [ ]:
df.to_csv('../processed_invoices1.csv', index=False)
print("Saved processed_invoices1.csv successfully!")
print(f"Dataframe successfully exported with shape: {df.shape}")